# TinyML mini-SE-Net (raw waveform)

Small 1-D CNN with a Squeeze-and-Excitation block that classifies **raw audio waveform** and exports to **int8 TFLite** for edge deployment. Added as a comparison model in the pipeline.

**Three changes from the original `build_mini_senet`:**
- `reduction_ratio` is now defined (was undefined → `NameError`).
- First conv uses a large strided kernel (was `kernel=3`) — gives a real receptive field and shrinks the sequence for the MCU.
- `LayerNorm` → `BatchNorm` (TFLite-Micro friendly; folds into the conv at conversion).

Reads the split file from stage `01` and writes a `run_<ts>` folder so `04_compare_results` picks it up.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
# ===== CELL 1: config ===============================================
import os, json, time, csv, datetime, hashlib
import numpy as np

DATASET       = "Multiclass_Drone_Audio"
SPLIT_DIR     = f"/content/drive/MyDrive/Final Project RMOT/processed/splits/{DATASET}"
SPLIT_FILE    = os.path.join(SPLIT_DIR, "processed_audio_splits.joblib")
RESULTS_DIR   = "/content/drive/MyDrive/Final Project RMOT/artifacts/results"
CACHE_DIR     = "/content/drive/MyDrive/Final Project RMOT/processed/raw_cache"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

SAMPLE_RATE   = 16000     # set to what 01/02 used. 8000 halves MCU cost; drone
                          # harmonics are mostly < 4 kHz so 8 kHz is usually fine.
CLIP_SECONDS  = 1.0
INPUT_LEN     = int(SAMPLE_RATE * CLIP_SECONDS)

REDUCTION_RATIO = 8       # SE bottleneck (32 ch -> 4 units)
FIRST_KERNEL    = 64
FIRST_STRIDE    = 8
EPOCHS          = 30
BATCH_SIZE      = 32
PATIENCE        = 6

MODEL_NAME = "mini_senet_raw"
FEATURE    = "raw_waveform"
AUGMENTED  = False

# names that count as the positive / "drone" class across datasets
POSITIVE_NAMES = {"drone", "yes_drone", "yes drone", "abnormal", "anomaly", "yes aircraft"}

In [ ]:
# ===== CELL 2: the model ============================================
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, MaxPooling1D,
                                      GlobalAveragePooling1D, Dense, Reshape, Multiply)
from tensorflow.keras.models import Model

def build_mini_senet(input_len, n_classes, reduction_ratio=REDUCTION_RATIO):
    inp = Input(shape=(input_len, 1))

    x = Conv1D(32, FIRST_KERNEL, strides=FIRST_STRIDE, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    feat = x
    ch   = feat.shape[-1]

    se = GlobalAveragePooling1D()(feat)
    se = Dense(max(1, ch // reduction_ratio), activation='relu')(se)
    se = Dense(ch, activation='sigmoid')(se)
    se = Reshape((1, ch))(se)
    x  = Multiply()([feat, se])

    x = Conv1D(32, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dense(16, activation='relu')(x)
    out = Dense(n_classes, activation='softmax')(x)
    return Model(inp, out)

In [ ]:
# ===== CELL 3: raw-audio loader (rides on the split file) ===========
import librosa, joblib

splits        = joblib.load(SPLIT_FILE)
unique_labels = splits["unique_labels"]
n_classes     = len(unique_labels)
pos_idx       = next((i for i, l in enumerate(unique_labels)
                      if l.lower() in POSITIVE_NAMES), 0)
print(f"classes={unique_labels} | positive='{unique_labels[pos_idx]}' (idx {pos_idx})")

def _load_clip(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y = librosa.util.fix_length(y, size=INPUT_LEN)   # crop/pad to exactly 1 s
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak                                  # per-clip peak norm (mic-level invariance)
    return y.astype(np.float32)

def load_split(paths, y, tag):
    key   = hashlib.md5(("|".join(paths) + f"{SAMPLE_RATE}_{INPUT_LEN}").encode()).hexdigest()[:12]
    cache = os.path.join(CACHE_DIR, f"{tag}_{key}.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return d["X"], d["y"]
    X = np.stack([_load_clip(p) for p in paths])[..., None]   # (N, INPUT_LEN, 1)
    y = np.asarray(y, dtype=np.int64)
    np.savez_compressed(cache, X=X, y=y)
    return X, y

X_train, y_train = load_split(splits["X_train"], splits["y_train"], "train")
X_val,   y_val   = load_split(splits["X_val"],   splits["y_val"],   "val")
X_test,  y_test  = load_split(splits["X_test"],  splits["y_test"],  "test")
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")

classes=['drone', 'normal'] | positive='drone' (idx 0)
train (8192, 16000, 1) | val (1756, 16000, 1) | test (1756, 16000, 1)


In [ ]:
# ===== CELL 4: train (class-weighted for imbalance) =================
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

present = np.unique(y_train)
cw      = compute_class_weight("balanced", classes=present, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(present, cw)}

model = build_mini_senet(INPUT_LEN, n_classes)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
n_params = model.count_params()
print(f"params: {n_params:,}")
model.summary()

train_start = time.perf_counter()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=3,
            factor=0.5
        )
    ],
    verbose=2,
)

train_time_s = time.perf_counter() - train_start

params: 7,350


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 16000, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2000, 32)  │      2,080 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2000, 32)  │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 1000, 32)  │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ max_pooling1d_2[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 4)         │        132 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │        160 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 32)     │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 1000, 32)  │          0 │ max_pooling1d_2[… │
│ (Multiply)          │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 1000, 32)  │      3,104 │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1000, 32)  │        128 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 500, 32)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ max_pooling1d_3[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      1,056 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 16)        │        528 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 2)         │         34 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,350 (28.71 KB)

 Trainable params: 7,222 (28.21 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/30
256/256 - 10s - 38ms/step - accuracy: 0.8099 - loss: 0.3957 - val_accuracy: 0.9021 - val_loss: 0.4271 - learning_rate: 0.0010
Epoch 2/30
256/256 - 2s - 8ms/step - accuracy: 0.8798 - loss: 0.2438 - val_accuracy: 0.9305 - val_loss: 0.1949 - learning_rate: 0.0010
Epoch 3/30
256/256 - 2s - 8ms/step - accuracy: 0.8955 - loss: 0.2159 - val_accuracy: 0.9169 - val_loss: 0.1845 - learning_rate: 0.0010
Epoch 4/30
256/256 - 2s - 8ms/step - accuracy: 0.9205 - loss: 0.1696 - val_accuracy: 0.8610 - val_loss: 0.2831 - learning_rate: 0.0010
Epoch 5/30
256/256 - 2s - 8ms/step - accuracy: 0.9218 - loss: 0.1693 - val_accuracy: 0.9277 - val_loss: 0.1683 - learning_rate: 0.0010
Epoch 6/30
256/256 - 2s - 8ms/step - accuracy: 0.9331 - loss: 0.1487 - val_accuracy: 0.9510 - val_loss: 0.1119 - learning_rate: 0.0010
Epoch 7/30
256/256 - 2s - 10ms/step - accuracy: 0.9412 - loss: 0.1317 - val_accuracy: 0.9630 - val_loss: 0.0937 - learning_rate: 0.0010
Epoch 8/30
256/256 - 2s - 9ms/step - accuracy: 0.948

In [ ]:
# ===== CELL 5: evaluate =============================================
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, accuracy_score, confusion_matrix)

t0     = time.perf_counter()
probs  = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
keras_ms = (time.perf_counter() - t0) * 1000 / len(X_test)

y_pred  = probs.argmax(1)
y_true  = y_test
y_score = probs[:, pos_idx]
is_pos  = (y_true == pos_idx).astype(int)
pred_pos = (y_pred == pos_idx).astype(int)

metrics = {
    "auc":       float(roc_auc_score(is_pos, y_score)) if is_pos.min() != is_pos.max() else float("nan"),
    "f1":        float(f1_score(is_pos, pred_pos, zero_division=0)),
    "precision": float(precision_score(is_pos, pred_pos, zero_division=0)),
    "recall":    float(recall_score(is_pos, pred_pos, zero_division=0)),
    "accuracy":  float(accuracy_score(y_true, y_pred)),
}
print("metrics:", json.dumps(metrics, indent=2))
print("confusion:\n", confusion_matrix(y_true, y_pred))

metrics: {
  "auc": 0.995018115942029,
  "f1": 0.8207547169811321,
  "precision": 0.7767857142857143,
  "recall": 0.87,
  "accuracy": 0.9783599088838268
}
confusion:
 [[  87   13]
 [  25 1631]]


In [ ]:
# ===== CELL 6: TinyML — int8 TFLite export + on-device numbers ======
def representative_dataset():
    idx = np.random.choice(len(X_train), min(200, len(X_train)), replace=False)
    for i in idx:
        yield [X_train[i:i+1].astype(np.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations            = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset   = representative_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type     = tf.int8
conv.inference_output_type    = tf.int8
tflite_model = conv.convert()
tflite_kb    = len(tflite_model) / 1024

# int8 inference + latency
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
scale, zp    = inp_d["quantization"]

def _q(x):  # float -> int8 per the input quant params
    return np.clip(np.round(x / scale + zp), -128, 127).astype(np.int8)

t0 = time.perf_counter()
for i in range(len(X_test)):
    interp.set_tensor(inp_d["index"], _q(X_test[i:i+1]))
    interp.invoke()
tflite_ms = (time.perf_counter() - t0) * 1000 / len(X_test)

print(f"TinyML: {n_params:,} params | int8 size {tflite_kb:.1f} KB | "
      f"{tflite_ms:.2f} ms/sample (int8) vs {keras_ms:.2f} ms (float keras)")

Saved artifact at '/tmp/tmpdcwoe4lx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 16000, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  138813459125392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459125584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459121552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459125200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459126352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459128656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459128464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459132880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459127696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813459120784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138813448161232: 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TinyML: 7,350 params | int8 size 22.6 KB | 1.32 ms/sample (int8) vs 0.73 ms (float keras)


In [ ]:
# ===== CELL 7: connect TinyML to the RMOT report pipeline ==========

import os
import sys
import subprocess


# Find reporting.py in the RMOT project
possible_reporting_dirs = [
    "/content/drive/MyDrive/Final Project RMOT/src",
    "/content/drive/MyDrive/Final Project RMOT/notebooks",
    "/content/drive/MyDrive/Final Project RMOT",
]

reporting_dir = next(
    (
        folder for folder in possible_reporting_dirs
        if os.path.exists(os.path.join(folder, "reporting.py"))
    ),
    None
)

if reporting_dir is None:
    raise FileNotFoundError(
        "Could not find reporting.py inside Final Project RMOT"
    )

if reporting_dir not in sys.path:
    sys.path.insert(0, reporting_dir)

from reporting import TrainingReport


# Convert the original labels into:
# 1 = drone/anomaly
# 0 = background/normal
y_train_binary = (np.asarray(y_train) == pos_idx).astype(int)
y_val_binary   = (np.asarray(y_val) == pos_idx).astype(int)
y_test_binary  = (np.asarray(y_test) == pos_idx).astype(int)

y_pred_binary = pred_pos
drone_scores  = y_score


# Create the same report object used by the rest of the pipeline
report = TrainingReport(
    root_dir=RESULTS_DIR,
    dataset=DATASET,
    augmented=AUGMENTED,

    # In this report: 1 means drone/anomaly
    anomaly_label=1,
    label_names=("background", "drone"),

    hyperparams={
        "sample_rate": SAMPLE_RATE,
        "clip_seconds": CLIP_SECONDS,
        "input_length": INPUT_LEN,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "patience": PATIENCE,
        "first_kernel": FIRST_KERNEL,
        "first_stride": FIRST_STRIDE,
        "reduction_ratio": REDUCTION_RATIO,
    },

    extra_metadata={
        "deployment": "TinyML / int8 TFLite",
        "parameters": int(n_params),
        "keras_latency_ms_sample": round(keras_ms, 3),
        "tflite_latency_ms_sample": round(tflite_ms, 3),
        "tflite_size_kb": round(tflite_kb, 1),
        "input_type": "raw waveform",
        "input_shape": list(model.input_shape),
        "positive_class": str(unique_labels[pos_idx]),
        "split_mode": splits.get("split_mode"),
    },
)


# Add dataset information to the report
report.log_data_summary(
    y_train_binary,
    y_val_binary,
    y_test_binary,
)


# TFLite total inference time
tflite_total_seconds = (
    tflite_ms * len(X_test) / 1000.0
)


# Add mini-SE-Net as a normal pipeline result
report.add_result(
    feature_name="raw_waveform",
    model_name="mini-SE-Net int8",

    y_true=y_test_binary,
    y_pred=y_pred_binary,

    # Continuous drone probability, used for ROC and PR curves
    scores=drone_scores,
    scores_higher_is_anomaly=True,

    train_time_s=train_time_s,

    # This makes the report show the int8 latency,
    # rather than the slower Keras latency
    latency_total_s=tflite_total_seconds,

    model_hyperparams={
        "parameters": int(n_params),
        "tflite_size_kb": round(tflite_kb, 1),
        "quantization": "full int8",
    },

    notes=(
        f"Full-int8 TFLite model; "
        f"{tflite_kb:.1f} KB; "
        f"{tflite_ms:.3f} ms/sample measured with the "
        f"TFLite interpreter on the Colab host, not yet on the target device."
    ),
)


# Save the actual deployable model inside the report folder
tflite_path = os.path.join(
    report.run_dir,
    "mini_senet_int8.tflite"
)

with open(tflite_path, "wb") as file:
    file.write(tflite_model)


# Generate CSV, JSON, figures and HTML
html_path = report.finalize()


# Convert HTML report to PDF
try:
    from weasyprint import HTML
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "weasyprint",
    ])
    from weasyprint import HTML

pdf_path = os.path.join(report.run_dir, "report.pdf")

HTML(
    filename=html_path,
    base_url=report.run_dir
).write_pdf(pdf_path)


print("Run folder:", report.run_dir)
print("HTML report:", html_path)
print("PDF report:", pdf_path)
print("TFLite model:", tflite_path)

saved run: /content/drive/MyDrive/Final Project RMOT/artifacts/results/run_20260621_132316_mini_senet_raw
